In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
import torch.nn as nn

In [3]:
class DynamicGraph(nn.Module):
    def __init__(self, feature_dim=64, max_nodes=1024, max_edges_per_node=8, input_size=64, output_size=2, hidden_layers=[128, 256, 256, 128, 64], device=None):
        super().__init__()
        if device==None:
            if hasattr(torch, "xpu") and torch.xpu.is_available():
                device = torch.device("xpu")
            elif torch.cuda.is_available():
                device = torch.device("cuda")
            else:
                device = torch.device("cpu")
        self.device = torch.device(device)
        print(f"\nUsing device : {device}\n")

        self.D = feature_dim
        self.max_nodes = max_nodes
        self.max_edges = max_nodes * max_edges_per_node
        # max_edges = self.max_edges
        self.max_edges_per_node = max_edges_per_node
        self.input_size = input_size
        self.output_size = output_size
        self.hidden_layers = hidden_layers
        self.max_rank = len(hidden_layers)

        ## Node Thingys

        # Node Weights (no_of_nodes, feature_dim)
        self.node_weight = nn.Parameter(
            torch.randn(
                max_nodes,
                feature_dim,
                device=self.device
            ) #* 0.02
        )

        # Node Pre Activation(Input of current node) (no_of_nodes,1)
        self.register_buffer(
            "node_pre_activation",
            torch.zeros(
                max_nodes,
                feature_dim,
                dtype=torch.float32,
                device=self.device
            )
        )

        # Node Activation(Input of current node) (no_of_nodes,1)
        self.register_buffer(
            "node_activation",
            torch.zeros(
                max_nodes,
                feature_dim,
                dtype=torch.float32,
                device=self.device
            )
        )

        # Node Bias (no_of_nodes,1)
        self.node_bias = nn.Parameter(
            torch.randn(
                max_nodes,
                feature_dim,
                device=self.device
            ) #* 0.02
        )

        # Node Active/Deactive (no_of_nodes,1)
        self.register_buffer(
            "node_alive",
            torch.zeros(
                max_nodes,
                dtype=torch.bool,
                device=self.device
            )
        )

        # Node Activation Type (Relu/Sigmoid/LRelu) (no_of_nodes,1)
        self.register_buffer(
            "node_activation_type",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )   
        )

        # Node Out Degree (no_of_nodes,1) (Value ranges from 0 to max_edges_per_node)
        self.register_buffer(
            "out_degree",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )
        )

        # Node In Degree (no_of_nodes,1) (Value ranges from 0 to no_of_nodes * max_edges_per_node)
        self.register_buffer(
            "in_degree",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )
        )

        # Node Rank (no_of_nodes,1)
        # Used to store the rank of nodes, which is used to prevent cycles
        self.register_buffer(
            "node_rank",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )
        )

        # No of current nodes (Includes Active + Deactive)
        self.nodes_count = 0


        ## Edge Thingys

        # Edge Source[edge_source][index] = edge_destination (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_src",
            -1*torch.ones(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Index[edge_destination][index] = edge_source (no_of_nodes,max_edges_per_node)
        # Used to store the index of edge stored in edge_src matrix for quick operation in back prop
        self.register_buffer(
            "edge_index",
            -1*torch.ones(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Destination[edge_destination][index] = edge_source (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_dest",
            -1*torch.ones(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Weight[edge_source][edge_destination] (no_of_nodes,max_edges_per_node)
        self.edge_weight = nn.Parameter(
            torch.randn(
                max_nodes,
                max_edges_per_node,
                device=self.device
            ) #* 0.02
        )

        # Edge Active/Deactive[edge_source][edge_destination] (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_alive",
            torch.zeros(
                max_nodes,
                max_edges_per_node,
                dtype=torch.bool,
                device=self.device
            )   
        )

    @torch.no_grad()
    def add_node(self, activation=1):
        if self.nodes_count >= self.max_nodes:
            raise RuntimeError("Node capacity reached in Add Node"+str(self.max_nodes))
        index = self.nodes_count
        self.node_alive[index] = True
        self.node_activation_type[index] = activation
        self.nodes_count += 1
        return index

    @torch.no_grad()
    def remove_node(self, index):
        if index>=self.nodes_count or index<0:
            raise RuntimeError("Out of bound in Remove Node")

        self.node_alive[index] = False

    @torch.no_grad()
    def add_edge(self, source, destination):
        if source<0 or source>=self.nodes_count or destination<0 or destination>=self.nodes_count:
            raise RuntimeError("Out of bound in Add Edge 1st Thingy")
        if self.out_degree[source]>=self.max_edges_per_node or self.in_degree[destination]>=self.max_edges_per_node:
            raise RuntimeError("Out of bound in Add Edge 2nd Thingy")
        self.edge_src[source][self.out_degree[source]] = destination

        self.edge_index[destination][self.in_degree[destination]] = self.out_degree[source]
        self.edge_dest[destination][self.in_degree[destination]] = source

        self.edge_alive[source][self.out_degree[source]] = True

        self.out_degree[source] += 1
        self.in_degree[destination] += 1

    @torch.no_grad()
    def remove_edge(self, source, destination):
        if source<0 or source>=self.nodes_count or destination<0 or destination>=self.nodes_count:
            raise RuntimeError("Out of bound in Remove Edge")
        temp_index = self.edge_src[source]==destination
        self.edge_alive[source][temp_index] = False

    @torch.no_grad()
    def random_sample_source(self, start_index, end_index, target_size, destination_idx=None):
        source_idx = torch.zeros(target_size,dtype=torch.long,device=self.device)
        temp_idx = torch.arange(0,self.max_nodes,1,device=self.device)

        temp_idx = temp_idx[start_index:end_index][self.in_degree[start_index:end_index]<self.max_edges_per_node-1]
        
        temp_idx = temp_idx[torch.randperm(len(temp_idx))]
        source_idx[:min(len(temp_idx),target_size)] = temp_idx[:target_size]

        start, end = 0, min(len(temp_idx),target_size)
        self.edge_src[source_idx[start: end], self.out_degree[source_idx[start: end]]] = destination_idx[start: end]
        self.edge_index[destination_idx[start: end], self.in_degree[destination_idx[start: end]]] = self.out_degree[source_idx[start: end]]
        self.edge_dest[destination_idx[start: end], self.in_degree[destination_idx[start: end]]] = source_idx[start: end]
        self.out_degree[source_idx[start: end]] += 1
        self.in_degree[destination_idx[start: end]] += 1
        
        current_size = min(len(temp_idx),target_size)
        current_target_size = target_size - current_size
        while current_size!=target_size:
            if current_size==0:
                print("Can't Initialize as no of nodes in some rank isn't enough, Please increase no of nodes appropriately!!")
                raise RuntimeError("Can't Initialize as no of nodes in some rank isn't enough, Please increase no of nodes appropriately!!")
            
            temp_idx = torch.arange(0,self.max_nodes,1,device=self.device)

            temp_idx = temp_idx[start_index:end_index][self.in_degree[start_index:end_index]<self.max_edges_per_node-1]
            
            temp_idx = temp_idx[torch.randperm(len(temp_idx))]
            if len(temp_idx) == 0:
                print("Can't Initialize as max edges per node isn't enough, Please increase max edges per node accordingly!!")
                raise RuntimeError("Can't Initialize as max edges per node isn't enough, Please increase max edges per node accordingly!!")
            start, end = current_size, current_size + min(current_target_size,len(temp_idx))
            source_idx[start: end] = temp_idx[:current_target_size]

            

            self.edge_src[source_idx[start: end], self.out_degree[source_idx[start: end]]] = destination_idx[start: end]
            self.edge_index[destination_idx[start: end], self.in_degree[destination_idx[start: end]]] = self.out_degree[source_idx[start: end]]
            self.edge_dest[destination_idx[start: end], self.in_degree[destination_idx[start: end]]] = source_idx[start: end]
            self.out_degree[source_idx[start: end]] += 1
            self.in_degree[destination_idx[start: end]] += 1

            current_size += min(current_target_size,len(temp_idx))
            current_target_size = target_size - current_size
            
        return source_idx
    
    @torch.no_grad()
    def random_sample_destination(self, start_index, end_index, target_size, source_idx=None):
        destination_idx = torch.zeros(target_size,dtype=torch.long,device=self.device)
        temp_idx = torch.arange(0,self.max_nodes,1,device=self.device)

        temp_idx = temp_idx[start_index:end_index][self.in_degree[start_index:end_index]<self.max_edges_per_node-1]
        
        temp_idx = temp_idx[torch.randperm(len(temp_idx))]
        destination_idx[:min(len(temp_idx),target_size)] = temp_idx[:target_size]

        start, end = 0, min(len(temp_idx),target_size)
        self.edge_src[source_idx[start: end], self.out_degree[source_idx[start: end]]] = destination_idx[start: end]
        self.edge_index[destination_idx[start: end], self.in_degree[destination_idx[start: end]]] = self.out_degree[source_idx[start: end]]
        self.edge_dest[destination_idx[start: end], self.in_degree[destination_idx[start: end]]] = source_idx[start: end]
        self.out_degree[source_idx[start: end]] += 1
        self.in_degree[destination_idx[start: end]] += 1
        
        current_size = min(len(temp_idx),target_size)
        current_target_size = target_size - current_size
        while current_size!=target_size:
            if current_size==0:
                print("Can't Initialize as no of nodes in some rank isn't enough, Please increase no of nodes appropriately!!")
                raise RuntimeError("Can't Initialize as no of nodes in some rank isn't enough, Please increase no of nodes appropriately!!")
            
            temp_idx = torch.arange(0,self.max_nodes,1,device=self.device)

            temp_idx = temp_idx[start_index:end_index][self.in_degree[start_index:end_index]<self.max_edges_per_node-1]
            
            temp_idx = temp_idx[torch.randperm(len(temp_idx))]
            if len(temp_idx) == 0:
                print("Can't Initialize as max edges per node isn't enough, Please increase max edges per node accordingly!!")
                raise RuntimeError("Can't Initialize as max edges per node isn't enough, Please increase max edges per node accordingly!!")
            start, end = current_size, current_size + min(current_target_size,len(temp_idx))
            destination_idx[start: end] = temp_idx[:current_target_size]

            

            self.edge_src[source_idx[start: end], self.out_degree[source_idx[start: end]]] = destination_idx[start: end]
            self.edge_index[destination_idx[start: end], self.in_degree[destination_idx[start: end]]] = self.out_degree[source_idx[start: end]]
            self.edge_dest[destination_idx[start: end], self.in_degree[destination_idx[start: end]]] = source_idx[start: end]
            self.out_degree[source_idx[start: end]] += 1
            self.in_degree[destination_idx[start: end]] += 1

            current_size += min(current_target_size,len(temp_idx))
            current_target_size = target_size - current_size
            
        return destination_idx

    ## Note: This Initialize Graph method can create edges between the same pair of nodes multiple times too.
    @torch.no_grad()
    def initialize_graph(self):
        total_layers = sum(self.hidden_layers)
        if total_layers>self.max_nodes:
            raise RuntimeError("More no of hidden thingys in Initialize Graph")
        
        current_count = 0
        temp_layers = self.hidden_layers.copy()
        print(temp_layers)
        output_layer_count = temp_layers.pop()
        print(temp_layers)
        for i in range(len(temp_layers)):
            layer_count = temp_layers[i]
            self.node_activation_type[current_count:current_count+layer_count] = 1
            self.node_rank[current_count:current_count+layer_count] = i+1
            current_count += layer_count
        self.node_activation_type[current_count:current_count+output_layer_count] = 2
        self.node_rank[current_count:current_count+output_layer_count] = len(self.hidden_layers)
        self.node_alive[:total_layers] = True
        self.nodes_count = total_layers

        current_count = 0
        for i in range(len(self.hidden_layers)-1):
            source_idx = torch.randperm(
                self.hidden_layers[i],
                device=self.device
            )
            source_idx += current_count

            destination_idx = self.random_sample_destination(
                self.hidden_layers[i]+current_count,
                total_layers,
                self.hidden_layers[i],
                source_idx=source_idx
            )
            current_count += self.hidden_layers[i]
            
        current_count = self.hidden_layers[0]
        for i in range(1,len(self.hidden_layers)):
            destination_idx = torch.randperm(
                self.hidden_layers[i],
                device=self.device
            )
            destination_idx += current_count
            print(destination_idx)

            source_idx = self.random_sample_source(
                0,
                current_count,
                self.hidden_layers[i],
                destination_idx=destination_idx
            )
            current_count += self.hidden_layers[i]

    def forward(self, x, cache_lrp=False):
        if cache_lrp:
            self.lrp_cache = {}
        self.node_pre_activation = self.node_pre_activation.detach()
        self.node_activation = self.node_activation.detach()
        rank_1_mask = (self.node_rank==1) & self.node_alive
        self.node_pre_activation[rank_1_mask] = x * self.node_weight[rank_1_mask] + self.node_bias[rank_1_mask]
        self.node_activation[rank_1_mask] = apply_activation(self.node_pre_activation[rank_1_mask],self.node_activation_type[rank_1_mask])
        n = len(self.edge_src)
        m = len(self.edge_src[0])
        temp_src_matrix = torch.arange(n, device=self.node_weight.device).unsqueeze(1).expand(n, m)
        temp_src_index = torch.arange(m, device=self.node_weight.device).repeat(n,1)
        for rank in range(2,self.max_rank+1):
            current_mask = (self.node_rank==rank) & self.node_alive
            prev_mask = (self.node_rank==(rank-1)) & self.node_alive
            self.node_pre_activation[current_mask] = 0
            # node_pre_activation[node_rank==rank] = node_pre_activation[node_rank==rank-1] * node_weight[node_rank==rank] + node_bias[node_rank==rank]
            valid_edges = self.edge_alive & prev_mask.unsqueeze(1)
            temp_src = temp_src_matrix[valid_edges]
            temp_index = temp_src_index[valid_edges]
            temp_dest = self.edge_src[valid_edges]
            edge_contribution = (self.node_activation[temp_src]*self.edge_weight[temp_src, temp_index].unsqueeze(1) * self.node_weight[temp_dest])
            self.node_pre_activation.index_add_(0,temp_dest,edge_contribution)
            self.node_pre_activation[current_mask] += self.node_bias[current_mask]
            self.node_activation[current_mask] = apply_activation(self.node_pre_activation[current_mask],self.node_activation_type[current_mask])

            if cache_lrp:

                self.lrp_cache[rank] = {
                    "temp_src": temp_src.detach().clone(),

                    "temp_dest": temp_dest.detach().clone(),

                    "temp_index": temp_index.detach().clone(),

                    "edge_contribution":
                        edge_contribution.detach().clone(),

                    "node_pre_activation":
                        self.node_pre_activation[current_mask]
                        .detach()
                        .clone(),

                    "node_activation":
                        self.node_activation[current_mask]
                        .detach()
                        .clone()
                }

        output = self.node_activation[(self.node_rank==self.max_rank) & self.node_alive]
        output = output.mean(dim=1)
        return output


    
    @torch.no_grad()
    def classwise_lrp(self, epsilon=1e-6):
        if len(self.lrp_cache) == 0:
            raise RuntimeError(
                "LRP cache is empty. "
                "Run forward(x, cache_lrp=True) first."
            )
    
        device = self.device
    
        D = self.D
        C = self.output_size
    
        # ---------------------------------------------------------
        # Output nodes
        # ---------------------------------------------------------
    
        output_nodes = torch.where(
            (self.node_rank == self.max_rank)
            & self.node_alive
        )[0]
    
        if len(output_nodes) != C:
            raise RuntimeError(
                f"Expected {C} output nodes, "
                f"but found {len(output_nodes)}."
            )
    
        # ---------------------------------------------------------
        # Node relevance
        #
        # [num_nodes, feature_dim, classes]
        # ---------------------------------------------------------
    
        node_relevance = torch.zeros(
            self.max_nodes,
            D,
            C,
            device=device
        )
    
        # ---------------------------------------------------------
        # Initialize relevance at output
        # ---------------------------------------------------------
    
        output_activation = self.node_activation[
            output_nodes
        ]
    
        # output_activation:
        #
        # [C, D]
        #
        # Each output class is:
        #
        # mean(output_activation[class])
        #
        # We initialize relevance proportional to
        # the feature contribution to that mean.
        # ---------------------------------------------------------
    
        for c in range(C):
    
            output_feature = output_activation[c]
    
            class_score = output_feature.mean()
    
            # Simple proportional initialization
            #
            # R_d = activation_d / sum(activation)
            #
            # multiplied by class score.
            #
            # epsilon prevents division by zero.
    
            denominator = (
                output_feature.sum()
                + epsilon
            )
    
            node_relevance[
                output_nodes[c],
                :,
                c
            ] = (
                output_feature
                / denominator
                * class_score
            )
    
        # ---------------------------------------------------------
        # Edge relevance
        #
        # We will store relevance for every edge and class.
        #
        # [max_nodes, max_edges_per_node, C]
        #
        # This corresponds to your edge slots.
        # ---------------------------------------------------------
    
        edge_relevance_all = torch.zeros(
            self.max_nodes,
            self.max_edges_per_node,
            C,
            device=device
        )
    
        # ---------------------------------------------------------
        # Propagate backwards through ranks
        # ---------------------------------------------------------
    
        for rank in range(
            self.max_rank,
            1,
            -1
        ):
    
            cache = self.lrp_cache[rank]
    
            temp_src = cache["temp_src"]
    
            temp_dest = cache["temp_dest"]
    
            temp_index = cache["temp_index"]
    
            z = cache["edge_contribution"]
    
            # -----------------------------------------------------
            # z shape:
            #
            # [E, D]
            #
            # We need:
            #
            # [E, D, C]
            #
            # -----------------------------------------------------
    
            z = z.unsqueeze(-1)
    
            # -----------------------------------------------------
            # Sum all incoming contributions for each destination
            #
            # denominator:
            #
            # [num_nodes, D]
            # -----------------------------------------------------
    
            denominator = torch.zeros(
                self.max_nodes,
                D,
                device=device
            )
    
            denominator.index_add_(
                0,
                temp_dest,
                z.squeeze(-1)
            )
    
            # -----------------------------------------------------
            # LRP epsilon stabilization
            # -----------------------------------------------------
    
            denominator = (
                denominator
                +
                epsilon * torch.sign(denominator)
            )
    
            # Handle exact zero
    
            denominator = torch.where(
                denominator.abs() < epsilon,
                torch.full_like(
                    denominator,
                    epsilon
                ),
                denominator
            )
    
            # -----------------------------------------------------
            # Get destination relevance
            #
            # [E, D, C]
            # -----------------------------------------------------
    
            destination_relevance = (
                node_relevance[temp_dest]
            )
    
            # -----------------------------------------------------
            # Denominator for every edge
            #
            # [E, D]
            # -----------------------------------------------------
    
            edge_denominator = (
                denominator[temp_dest]
            )
    
            # -----------------------------------------------------
            # Convert:
            #
            # z / denominator
            #
            # [E, D, 1]
            #
            # -----------------------------------------------------
    
            redistribution_ratio = (
                z
                /
                edge_denominator.unsqueeze(-1)
            )
    
            # -----------------------------------------------------
            # Edge feature-wise relevance
            #
            # [E, D, C]
            # -----------------------------------------------------
    
            edge_relevance_feature = (
                redistribution_ratio
                *
                destination_relevance
            )
    
            # -----------------------------------------------------
            # Convert feature-wise relevance to scalar edge
            # relevance
            #
            # [E, C]
            # -----------------------------------------------------
    
            edge_relevance = (
                edge_relevance_feature.sum(dim=1)
            )
    
            # -----------------------------------------------------
            # Store relevance for this edge
            # -----------------------------------------------------
    
            edge_relevance_all[
                temp_src,
                temp_index
            ] += edge_relevance
    
            # -----------------------------------------------------
            # Propagate relevance to source nodes
            # -----------------------------------------------------
    
            node_relevance_next = torch.zeros_like(
                node_relevance
            )
    
            node_relevance_next.index_add_(
                0,
                temp_src,
                edge_relevance_feature
            )
    
            # -----------------------------------------------------
            # Keep relevance from earlier ranks that hasn't
            # been processed yet
            # -----------------------------------------------------
    
            node_relevance = node_relevance_next
    
        return edge_relevance_all

    # ============================================================
    # PHASE 7: GRAPH SNAPSHOTS AND VALIDATION
    # Paste inside DynamicGraph
    # ============================================================

    @torch.no_grad()
    def _graph_snapshot(self):
        """Save graph structure and parameters for exact rollback."""
        names = [
            "node_weight", "node_bias", "node_alive",
            "node_activation_type", "node_rank",
            "out_degree", "in_degree",
            "edge_src", "edge_index", "edge_dest",
            "edge_weight", "edge_alive",
            "node_pre_activation", "node_activation"
        ]

        return {
            "tensors": {
                name: getattr(self, name).detach().clone()
                for name in names
            },
            "nodes_count": self.nodes_count
        }

    @torch.no_grad()
    def _restore_graph(self, snapshot):
        """Restore a graph snapshot without replacing Parameters."""
        for name, value in snapshot["tensors"].items():
            getattr(self, name).copy_(value)

        self.nodes_count = snapshot["nodes_count"]

    
    @torch.no_grad()
    def _node_importance(self):
        """Estimate normalized node importance."""
        importance = self.node_activation.detach().abs().mean(dim=1)

        importance = importance * self.node_alive.float()
        importance = torch.nan_to_num(
            importance, nan=0.0, posinf=0.0, neginf=0.0
        )

        total = importance.sum()
        if total > 0:
            importance = importance / total

        return importance

    @torch.no_grad()
    def _edge_importance(self):
        """Estimate edge importance from its contribution and weight."""
        scores = torch.zeros_like(self.edge_weight)

        for source in range(self.nodes_count):
            valid = self.edge_alive[source]
            if not valid.any():
                continue

            destinations = self.edge_src[source, valid]
            slots = torch.where(valid)[0]

            scores[source, slots] = (
                self.node_activation[source].abs().mean()
                * self.edge_weight[source, slots].abs()
                * self.node_weight[destinations].abs().mean(dim=1)
            )

        scores *= self.edge_alive.float()

        total = scores.sum()
        if total > 0:
            scores /= total

        return scores


    @torch.no_grad()
    def _evaluate_graph(self, validation_batches):
        """
        Evaluate a fixed graph on reusable validation batches.

        Supports:
          X: [features] or [batch, features]
          y: integer labels or one-hot labels

        forward() is called one sample at a time because the
        existing model stores activations in shared node buffers.
        """
        was_training = self.training
        self.eval()

        all_scores = []
        all_labels = []

        try:
            for X, y in validation_batches:
                X = torch.as_tensor(X, device=self.device)
                y = torch.as_tensor(y, device=self.device)

                if X.ndim == 1:
                    X = X.unsqueeze(0)

                # Accept either integer labels or one-hot labels.
                if y.ndim > 1 and y.shape[-1] == self.output_size:
                    y = y.argmax(dim=-1)
                y = y.long().reshape(-1)

                if X.shape[0] != y.shape[0]:
                    raise ValueError(
                        f"Batch mismatch: X has {X.shape[0]} samples, "
                        f"but y has {y.shape[0]} labels."
                    )

                for i in range(X.shape[0]):
                    scores = self.forward(X[i], cache_lrp=True)

                    if scores.ndim != 1 or scores.numel() != self.output_size:
                        raise ValueError(
                            "Expected forward(sample) to return "
                            f"[{self.output_size}] scores; got "
                            f"{tuple(scores.shape)}."
                        )

                    all_scores.append(scores.float())
                    all_labels.append(y[i])

            if not all_scores:
                raise ValueError("Validation batches are empty.")

            scores = torch.stack(all_scores)
            labels = torch.stack(all_labels)

            predictions = scores.argmax(dim=1)

            accuracy = (
                predictions == labels
            ).float().mean().item()

            loss = torch.nn.functional.cross_entropy(
                scores, labels
            ).item()

            # Macro F1 across all output classes.
            f1_per_class = []

            for c in range(self.output_size):
                tp = ((predictions == c) & (labels == c)).sum().float()
                fp = ((predictions == c) & (labels != c)).sum().float()
                fn = ((predictions != c) & (labels == c)).sum().float()

                precision = tp / (tp + fp).clamp_min(1)
                recall = tp / (tp + fn).clamp_min(1)

                f1 = (
                    2 * precision * recall
                    / (precision + recall).clamp_min(1e-12)
                )
                f1_per_class.append(f1)

            macro_f1 = torch.stack(f1_per_class).mean().item()

            confidence = torch.softmax(
                scores, dim=1
            ).max(dim=1).values.mean().item()

            # A concentration proxy from node importance.
            # It is true LRP only if classwise_lrp() works correctly.
            try:
                node_scores = self._node_importance()
                node_scores = node_scores[self.node_alive]
                total_importance = node_scores.sum()

                if node_scores.numel() and total_importance > 0:
                    concentration = (
                        node_scores.max()
                        / total_importance.clamp_min(1e-12)
                    ).item()
                else:
                    concentration = 0.0

            except (RuntimeError, KeyError, IndexError):
                concentration = float("nan")

            return {
                "accuracy": accuracy,
                "loss": loss,
                "f1": macro_f1,
                "confidence": confidence,
                "lrp_concentration": concentration
            }

        finally:
            self.train(was_training)


    @torch.no_grad()
    def _try_candidate(self, candidate, validation_batches):
        """
        Apply one proposed change temporarily, evaluate it, then
        restore the original graph. Returns its metrics.
        """
        snapshot = self._graph_snapshot()

        try:
            self._apply_candidate(candidate)
            metrics = self._evaluate_graph(validation_batches)
        except (RuntimeError, ValueError, IndexError):
            metrics = None
        finally:
            self._restore_graph(snapshot)

        return metrics

    @torch.no_grad()
    def _apply_candidate(self, candidate):
        """Apply one candidate using existing graph operations."""
        kind = candidate["type"]

        if kind == "mask_edge":
            source, destination = candidate["edge"]
            self.remove_edge(source, destination)

        elif kind == "strengthen_edge":
            source, slot = candidate["edge"]
            self.edge_weight[source, slot] *= candidate.get(
                "factor", 1.10
            )

        elif kind == "shortcut":
            source = candidate["source"]
            destination = candidate["destination"]

            self.add_edge(source, destination)

            slot = int(self.out_degree[source].item()) - 1
            self.edge_weight[source, slot] = candidate.get(
                "weight",
                0.1
            )

        elif kind == "expand_node":
            source = candidate["source"]
            target = candidate["target"]
            rank = int(self.node_rank[target].item())

            if rank <= 1 or rank >= self.max_rank:
                raise RuntimeError(
                    "Node expansion requires an internal hidden rank."
                )

            new_node = self.add_node(
                int(self.node_activation_type[target].item())
            )

            self.node_rank[new_node] = rank

            # Initialize from the important node, with small noise.
            self.node_weight[new_node].copy_(
                self.node_weight[target]
                + 0.01 * torch.randn_like(self.node_weight[target])
            )
            self.node_bias[new_node].copy_(self.node_bias[target])

            # Incoming connection: preceding rank -> new node.
            self.add_edge(source, new_node)

            # Outgoing connection: new node -> following rank.
            self.add_edge(new_node, candidate["destination"])

            self.edge_weight[source, self.out_degree[source] - 1] = 0.1
            self.edge_weight[new_node, self.out_degree[new_node] - 1] = 0.1

        else:
            raise ValueError(f"Unknown candidate type: {kind}")

    # ============================================================
    # PHASE 7: CANDIDATE GENERATION AND GRAPH SELECTION
    # ============================================================


    @torch.no_grad()
    def generate_candidates(
        self,
        max_candidates=8,
        low_utility_threshold=0.02,
        high_utility_threshold=0.15,
        high_lrp_threshold=0.15,
        correlation_threshold=0.90,
        shortcut_samples=128,
        exploration_probability=0.25
    ):
        """
        Stochastic greedy local search.

        Complexity is driven by active edges and a bounded number
        of sampled shortcut pairs, not all node pairs.
        """
        candidates = []

        active_nodes = torch.where(self.node_alive)[0]
        if active_nodes.numel() == 0:
            return candidates

        edge_scores = self._edge_importance()
        node_scores = self._node_importance()

        # Collect existing active edges in one operation.
        valid = (
            self.edge_alive
            & (self.edge_src >= 0)
            & self.node_alive.unsqueeze(1)
        )

        sources, slots = torch.where(valid)

        if sources.numel():
            destinations = self.edge_src[sources, slots]
            destination_alive = self.node_alive[destinations]

            sources = sources[destination_alive]
            slots = slots[destination_alive]
            destinations = destinations[destination_alive]

            utilities = edge_scores[sources, slots]

            # Strong edges: occasionally propose strengthening.
            strong_count = min(
                max(1, max_candidates // 4),
                sources.numel()
            )

            strong_idx = torch.topk(
                utilities, k=strong_count
            ).indices

            for i in strong_idx.tolist():
                if (
                    utilities[i].item() >= high_utility_threshold
                    and torch.rand((), device=self.device).item()
                    < exploration_probability
                ):
                    candidates.append({
                        "type": "strengthen_edge",
                        "edge": (
                            int(sources[i].item()),
                            int(slots[i].item())
                        ),
                        "factor": 1.10,
                        "utility": float(utilities[i].item())
                    })

            # Weak edges: prioritize pruning the least influential.
            weak_count = min(
                max(1, max_candidates // 2),
                sources.numel()
            )

            weak_idx = torch.topk(
                utilities, k=weak_count, largest=False
            ).indices

            for i in weak_idx.tolist():
                utility = float(utilities[i].item())

                if utility <= low_utility_threshold:
                    candidates.append({
                        "type": "mask_edge",
                        "edge": (
                            int(sources[i].item()),
                            int(destinations[i].item())
                        ),
                        "utility": utility
                    })

        # --------------------------------------------------------
        # Shortcuts: sample a bounded number of node pairs.
        # Only adjacent ranks are compatible with existing forward().
        # --------------------------------------------------------
        active_by_rank = {}

        for rank in range(1, self.max_rank + 1):
            active_by_rank[rank] = torch.where(
                self.node_alive & (self.node_rank == rank)
            )[0]

        for rank in range(1, self.max_rank):
            left = active_by_rank[rank]
            right = active_by_rank[rank + 1]

            if left.numel() == 0 or right.numel() == 0:
                continue

            # Randomly sample from high-importance nodes most of
            # the time, while preserving some exploration.
            def choose_pool(indices, limit):
                if indices.numel() <= limit:
                    return indices

                scores = node_scores[indices]
                if torch.rand((), device=self.device).item() < exploration_probability:
                    chosen = torch.randperm(
                        indices.numel(), device=self.device
                    )[:limit]
                else:
                    # Small randomized perturbation avoids always
                    # selecting exactly the same top nodes.
                    noisy_scores = scores + (
                        torch.rand_like(scores) * 0.05
                    )
                    chosen = torch.topk(
                        noisy_scores, k=limit
                    ).indices

                return indices[chosen]

            pool_size = max(
                2,
                int(shortcut_samples ** 0.5)
            )

            left_pool = choose_pool(left, pool_size)
            right_pool = choose_pool(right, pool_size)

            pair_count = min(
                shortcut_samples,
                left_pool.numel() * right_pool.numel()
            )

            # Sample pair indices, rather than enumerate the Cartesian product.
            li = torch.randint(
                left_pool.numel(),
                (pair_count,),
                device=self.device
            )
            ri = torch.randint(
                right_pool.numel(),
                (pair_count,),
                device=self.device
            )

            sampled_sources = left_pool[li]
            sampled_destinations = right_pool[ri]

            for source_t, destination_t in zip(
                sampled_sources, sampled_destinations
            ):
                source = int(source_t.item())
                destination = int(destination_t.item())

                if source == destination:
                    continue

                if self.out_degree[source] >= self.max_edges_per_node:
                    continue

                if self.in_degree[destination] >= self.max_edges_per_node:
                    continue

                already_exists = (
                    self.edge_alive[source]
                    & (self.edge_src[source] == destination)
                ).any()

                if already_exists:
                    continue

                # Avoid expensive pairwise correlation matrices.
                a = self.node_activation[source]
                b = self.node_activation[destination]

                correlation = torch.nn.functional.cosine_similarity(
                    a.unsqueeze(0),
                    b.unsqueeze(0),
                    dim=1,
                    eps=1e-8
                ).item()

                if correlation >= correlation_threshold:
                    candidates.append({
                        "type": "shortcut",
                        "source": source,
                        "destination": destination,
                        "weight": 0.1,
                        "utility": correlation
                    })

        # High-potential candidates first, then randomized tie-breaking.
        # Return only a bounded number for validation.
        if candidates:
            scores = torch.tensor(
                [c["utility"] for c in candidates],
                device=self.device
            )
            scores = scores + 0.01 * torch.rand_like(scores)

            order = torch.argsort(scores, descending=True).tolist()
            candidates = [candidates[i] for i in order]

        # Remove duplicate proposals for the same modification.
        unique = []
        seen = set()

        for candidate in candidates:
            key = (
                candidate["type"],
                candidate.get("edge"),
                candidate.get("source"),
                candidate.get("destination")
            )

            if key not in seen:
                seen.add(key)
                unique.append(candidate)

            if len(unique) >= max_candidates:
                break

        return unique

    @torch.no_grad()
    def optimize_topology(
        self,
        validation_batches,
        max_candidates=8,
        accuracy_delta=0.002,
        max_active_nodes=None,
        max_active_edges=None,
        require_lower_lrp_concentration=True
    ):
        """
        Evaluate candidate graphs and commit at most one change.

        accuracy_delta=0.002 means +0.2 percentage points.

        Validation batches must be reusable (e.g. a list), because
        every candidate is evaluated on the same data.
        """
        validation_batches = list(validation_batches)

        if not validation_batches:
            raise ValueError("Provide non-empty validation batches.")

        baseline = self._evaluate_graph(validation_batches)

        baseline_nodes = int(self.node_alive.sum().item())
        baseline_edges = int(self.edge_alive.sum().item())

        candidates = self.generate_candidates(
            max_candidates=max_candidates
        )

        results = []

        for candidate in candidates:
            trial = self._try_candidate(
                candidate, validation_batches
            )

            if trial is None:
                continue

            trial_nodes = baseline_nodes
            trial_edges = baseline_edges

            if candidate["type"] == "expand_node":
                trial_nodes += 1
                trial_edges += 2

            elif candidate["type"] == "shortcut":
                trial_edges += 1

            elif candidate["type"] == "mask_edge":
                trial_edges -= 1

            # Respect compute/size budgets.
            if (
                max_active_nodes is not None
                and trial_nodes > max_active_nodes
            ):
                continue

            if (
                max_active_edges is not None
                and trial_edges > max_active_edges
            ):
                continue

            accuracy_gain = trial["accuracy"] - baseline["accuracy"]
            loss_gain = baseline["loss"] - trial["loss"]
            f1_gain = trial["f1"] - baseline["f1"]
            lrp_gain = (
                baseline["lrp_concentration"]
                - trial["lrp_concentration"]
            )

            # Allow small metric fluctuations so topology can explore.
            accepted = (
                trial["accuracy"] >= baseline["accuracy"] - 0.05
                and trial["loss"] <= baseline["loss"] * 1.01
            )

            results.append({
                "candidate": candidate,
                "metrics": trial,
                "accepted": accepted,
                "accuracy_gain": accuracy_gain,
                "loss_gain": loss_gain,
                "f1_gain": f1_gain,
                "lrp_gain": lrp_gain
            })

        # Select the best candidate that satisfies all conditions.
        eligible = [r for r in results if r["accepted"]]

        if not eligible:
            return {
                "accepted": False,
                "reason": "No candidate satisfied the acceptance criteria.",
                "baseline": baseline,
                "candidates_tested": len(results)
            }

        best = max(
            eligible,
            key=lambda r: (
                r["metrics"]["f1"],
                r["metrics"]["accuracy"],
                -r["metrics"]["loss"]
            )
        )

        # Commit the selected change.
        self._apply_candidate(best["candidate"])

        return {
            "accepted": True,
            "modification": best["candidate"],
            "baseline": baseline,
            "selected_metrics": best["metrics"],
            "accuracy_gain": best["accuracy_gain"],
            "loss_gain": best["loss_gain"],
            "f1_gain": best["f1_gain"],
            "lrp_concentration_gain": best["lrp_gain"],
            "candidates_tested": len(results)
        }

def apply_activation(x, activation_type):

    output = torch.empty_like(x)

    # ReLU
    mask = activation_type == 0
    output[mask] = torch.relu(x[mask])

    # Sigmoid
    mask = activation_type == 1
    output[mask] = torch.sigmoid(x[mask])

    # Leaky ReLU
    mask = activation_type == 2
    output[mask] = torch.nn.functional.leaky_relu(
        x[mask],
        negative_slope=0.01
    )

    # Tanh
    mask = activation_type == 3
    output[mask] = torch.tanh(x[mask])

    # No activation
    mask = activation_type == 4
    output[mask] = x[mask]

    return output


In [4]:
import os
import pandas as pd
import torch

from PIL import Image
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_ROOT = r"/kaggle/input/datasets/jashwanth124/dl-dataset"

CSV_PATH = r"/kaggle/input/datasets/sachcithgn/dl-210-images-per-class-gnn/Train_210_filt.csv"

IMAGE_SIZE = 64
NUM_CHANNELS = 3
NUM_NODES = IMAGE_SIZE * IMAGE_SIZE

NUM_CLASSES = 43

BATCH_SIZE = 1

TEST_SIZE = 0.2
RANDOM_STATE = 42


# ============================================================
# DEVICE
# ============================================================

device = (
    torch.device("xpu")
    if hasattr(torch, "xpu") and torch.xpu.is_available()
    else torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

# device = "cpu"

print("Device:", device)


# ============================================================
# READ CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

# ============================================================
# KEEP FIRST 5 IMAGES PER CLASS
# ============================================================

df = (
    df.groupby("ClassId", sort=False, group_keys=False)
      .head(5)
      .reset_index(drop=True)
)

print("\nImages after limiting each class to 5:")
print("Total images:", len(df))
print("Number of classes:", df["ClassId"].nunique())
print(df["ClassId"].value_counts().sort_index())

print("\nTotal images:", len(df))
print("Number of classes:", df["ClassId"].nunique())

print("\nClass distribution:")
print(df["ClassId"].value_counts().sort_index())


# ============================================================
# CREATE CLASS-ID → 0...42 MAPPING
# ============================================================

class_ids = sorted(df["ClassId"].unique())

if len(class_ids) != NUM_CLASSES:
    raise RuntimeError(
        f"Expected {NUM_CLASSES} classes, "
        f"but found {len(class_ids)} classes."
    )

class_to_index = {
    class_id: index
    for index, class_id in enumerate(class_ids)
}

print("\nClass mapping:")
print(class_to_index)


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["ClassId"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTraining images:", len(train_df))
print("Testing images :", len(test_df))


# ============================================================
# IMAGE TRANSFORM
# ============================================================

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])


# ============================================================
# LOAD ALL IMAGES INTO TENSORS
# ============================================================

def load_images(dataframe, root_dir):

    images = []
    labels = []

    for index, row in dataframe.iterrows():

        # ----------------------------------------------------
        # Image path
        # ----------------------------------------------------

        relative_path = str(row["Path"])

        image_path = os.path.join(
            root_dir,
            relative_path
        )

        # ----------------------------------------------------
        # Load RGB image
        # ----------------------------------------------------

        image = Image.open(
            image_path
        ).convert("RGB")

        # ----------------------------------------------------
        # Resize + ToTensor
        # ----------------------------------------------------

        image = transform(image)

        # ----------------------------------------------------
        # [3, 64, 64]
        #
        # ->
        #
        # [64, 64, 3]
        # ----------------------------------------------------

        image = image.permute(
            1, 2, 0
        )

        # ----------------------------------------------------
        # [64, 64, 3]
        #
        # ->
        #
        # [4096, 3]
        # ----------------------------------------------------

        image = image.reshape(
            NUM_NODES,
            NUM_CHANNELS
        )

        images.append(image)

        # ----------------------------------------------------
        # Label
        # ----------------------------------------------------

        class_id = int(row["ClassId"])

        # Convert original ClassId to 0...42
        class_index = class_to_index[class_id]

        # ----------------------------------------------------
        # One-hot encode
        #
        # Example:
        #
        # class_index = 5
        #
        # [0,0,0,0,0,1,0,...,0]
        #
        # Shape = [43]
        # ----------------------------------------------------

        label = torch.zeros(
            NUM_CLASSES,
            dtype=torch.float32
        )

        label[class_index] = 1.0

        labels.append(label)

    # --------------------------------------------------------
    # Stack everything
    # --------------------------------------------------------

    images = torch.stack(images)

    # [N, 43]
    labels = torch.stack(labels)

    return images, labels


# ============================================================
# LOAD TRAINING DATA
# ============================================================

print("\nLoading training images...")

train_images, train_labels = load_images(
    train_df,
    DATASET_ROOT
)

print("Training images loaded.")


# ============================================================
# LOAD TESTING DATA
# ============================================================

print("\nLoading testing images...")

test_images, test_labels = load_images(
    test_df,
    DATASET_ROOT
)

print("Testing images loaded.")


# ============================================================
# PRINT EXACT TENSOR SHAPES
# ============================================================

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)

print("Train images :", train_images.shape)
print("Train labels :", train_labels.shape)

print("Test images  :", test_images.shape)
print("Test labels  :", test_labels.shape)

print("=" * 70)


# ============================================================
# CREATE TENSORDATASETS
# ============================================================

train_dataset = TensorDataset(
    train_images,
    train_labels
)

test_dataset = TensorDataset(
    test_images,
    test_labels
)


# ============================================================
# CREATE TRAIN LOADER
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


# ============================================================
# CREATE TEST LOADER
# ============================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# TEST DATALOADER
# ============================================================

x, y = next(iter(train_loader))

print("\n" + "=" * 70)
print("DATALOADER CHECK")
print("=" * 70)

print("Images :", train_images.shape)
print("Labels :", train_labels.shape)

print("Batch x:", x.shape)
print("Batch y:", y.shape)

print("\nFirst label:")
print(y[0])

print("\nLabel sum:")
print(y[0].sum())

print("=" * 70)

Device: cuda

Images after limiting each class to 5:
Total images: 215
Number of classes: 43
ClassId
0     5
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
11    5
12    5
13    5
14    5
15    5
16    5
17    5
18    5
19    5
20    5
21    5
22    5
23    5
24    5
25    5
26    5
27    5
28    5
29    5
30    5
31    5
32    5
33    5
34    5
35    5
36    5
37    5
38    5
39    5
40    5
41    5
42    5
Name: count, dtype: int64

Total images: 215
Number of classes: 43

Class distribution:
ClassId
0     5
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
11    5
12    5
13    5
14    5
15    5
16    5
17    5
18    5
19    5
20    5
21    5
22    5
23    5
24    5
25    5
26    5
27    5
28    5
29    5
30    5
31    5
32    5
33    5
34    5
35    5
36    5
37    5
38    5
39    5
40    5
41    5
42    5
Name: count, dtype: int64

Class mapping:
{np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64

In [5]:
EPOCHS = 2

In [6]:
# ============================================================
# MODEL SUMMARY + TRAINING + TESTING
# ============================================================

import os
import time
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from torchinfo import summary

model = DynamicGraph(
    feature_dim=3,
    max_nodes=50000,
    max_edges_per_node=20,
    input_size=4096,
    output_size=43,
    hidden_layers=[4096, 3000, 1500, 1000, 500, 300, 43],
    device=device
)

model.initialize_graph()

# ============================================================
# MODEL SUMMARY
# ============================================================

# print("\n" + "=" * 80)
# print("MODEL SUMMARY")
# print("=" * 80)

# summary(
#     model,
#     input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE),
#     device=device,
#     col_names=[
#         "input_size",
#         "output_size",
#         "num_params",
#         "trainable"
#     ],
#     depth=10
# )

# print("=" * 80)


# ============================================================
# LOSS + OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


# ============================================================
# TRAINING CONFIG
# ============================================================



best_test_accuracy = 0.0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": [],
    "train_images_per_sec": [],
    "test_images_per_sec": [],
}



# ============================================================
# DYNAMIC GRAPH OPTIMIZATION CONFIG
# Add after your existing history dictionary
# ============================================================

TOPOLOGY_OPTIMIZE_EVERY = 1
TOPOLOGY_MAX_CANDIDATES = 4
TOPOLOGY_ACCURACY_DELTA = 0.000

print(TOPOLOGY_OPTIMIZE_EVERY)
# Use a small, fixed subset for topology selection.
# Increase only after confirming runtime and memory usage.
TOPOLOGY_VALIDATION_BATCHES = 8

history["topology_changes"] = []
history["active_nodes"] = []
history["active_edges"] = []

best_test_accuracy = 0.0

# Prepare a fixed validation subset without changing either loader.
# Your current labels are one-hot vectors, so convert them to
# class indices for the topology evaluator.
topology_validation_batches = []

for images, labels in test_loader:
    images = images.squeeze(0)
    labels = labels.squeeze(0)

    topology_validation_batches.append((
        images.detach().cpu().unsqueeze(0),
        labels.detach().argmax().reshape(1).cpu()
    ))

    if len(topology_validation_batches) >= TOPOLOGY_VALIDATION_BATCHES:
        break



# ============================================================
# TRAINING LOOP
# ============================================================

print("\n" + "=" * 80)
print("TRAINING")
print("=" * 80)

for epoch in range(EPOCHS):

    model.train()

    epoch_start = time.perf_counter()

    running_loss = 0.0
    correct = 0
    total = 0

    train_start = time.perf_counter()

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1:02d}/{EPOCHS}",
        unit="batch",
        dynamic_ncols=True
    )

    for images, labels in progress:

        images = images.to(device)
        images = images.squeeze(0)
        labels = labels.to(device)
        labels = labels.squeeze(0)

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)

        loss = criterion(outputs, labels)

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        loss.backward()

        optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        batch_size = images.size(0)

        running_loss += loss.item()
        
        predicted = torch.argmax(outputs)
        target = torch.argmax(labels)
        
        total += 1
        correct += (predicted == target).item()
        
        current_loss = running_loss / total
        current_accuracy = 100.0 * correct / total

        elapsed = time.perf_counter() - train_start

        images_per_sec = total / max(elapsed, 1e-8)

        # Update progress bar
        progress.set_postfix(
            loss=f"{current_loss:.4f}",
            acc=f"{current_accuracy:.2f}%",
            img_s=f"{images_per_sec:.1f}"
        )

    # --------------------------------------------------------
    # Training statistics
    # --------------------------------------------------------

    train_loss = running_loss / total
    train_accuracy = 100.0 * correct / total

    train_elapsed = time.perf_counter() - train_start
    train_images_per_sec = total / max(train_elapsed, 1e-8)


    # ========================================================
    # TESTING
    # ========================================================

    model.eval()

    test_start = time.perf_counter()

    test_loss_total = 0.0
    test_correct = 0
    test_total = 0

    test_progress = tqdm(
        test_loader,
        desc="Testing",
        unit="batch",
        dynamic_ncols=True
    )

    with torch.no_grad():

        for images, labels in test_progress:

            images = images.to(device)
            images = images.squeeze(0)
            labels = labels.to(device)
            labels = labels.squeeze(0)

            outputs = model(images)

            loss = criterion(outputs, labels)

            # --------------------------------------------------------
            # Statistics
            # --------------------------------------------------------
            
            # One-hot label [43] → class index
            target = torch.argmax(labels)
            
            # Model output [43] → predicted class
            predicted = torch.argmax(outputs)
            
            test_loss_total += loss.item()
            
            test_total += 1
            
            test_correct += (predicted == target).item()
            
            current_test_loss = (
                test_loss_total / test_total
            )
            
            current_test_accuracy = (
                100.0 * test_correct / test_total
            )

            elapsed = time.perf_counter() - test_start

            test_images_per_sec = (
                test_total / max(elapsed, 1e-8)
            )

            test_progress.set_postfix(
                loss=f"{current_test_loss:.4f}",
                acc=f"{current_test_accuracy:.2f}%",
                img_s=f"{test_images_per_sec:.1f}"
            )


    # --------------------------------------------------------
    # Testing statistics
    # --------------------------------------------------------

    test_loss = test_loss_total / test_total

    test_accuracy = (
        100.0 * test_correct / test_total
    )

    test_elapsed = time.perf_counter() - test_start

    test_images_per_sec = (
        test_total / max(test_elapsed, 1e-8)
    )

    epoch_elapsed = time.perf_counter() - epoch_start

    
    # ========================================================
    # PHASE 7 + 8: ADAPTIVE TOPOLOGY OPTIMIZATION
    # ========================================================

    topology_result = None

    if (
        TOPOLOGY_OPTIMIZE_EVERY > 0
        and (epoch + 1) % TOPOLOGY_OPTIMIZE_EVERY == 0
    ):
        print("\nRunning adaptive topology optimization...")

        topology_result = model.optimize_topology(
            validation_batches=topology_validation_batches,
            max_candidates=TOPOLOGY_MAX_CANDIDATES,
            accuracy_delta=TOPOLOGY_ACCURACY_DELTA,
            max_active_nodes=model.max_nodes,
            max_active_edges=model.max_edges
        )

        if topology_result["accepted"]:
            modification = topology_result["modification"]

            history["topology_changes"].append({
                "epoch": epoch + 1,
                "modification": modification
            })

            print(
                f"Accepted topology modification: {modification}"
            )
            print(
                "Validation accuracy gain: "
                f"{topology_result['accuracy_gain'] * 100:.3f}%"
            )
        else:
            print(
                "No topology change accepted: "
                f"{topology_result['reason']}"
            )

    # Track current graph size.
    history["active_nodes"].append(
        int(model.node_alive.sum().item())
    )

    history["active_edges"].append(
        int(model.edge_alive.sum().item())
    )



    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)

    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    history["train_images_per_sec"].append(
        train_images_per_sec
    )

    history["test_images_per_sec"].append(
        test_images_per_sec
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if test_accuracy > best_test_accuracy:

        best_test_accuracy = test_accuracy

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        best_marker = " ★ BEST"

    else:

        best_marker = ""


    # ========================================================
    # EPOCH SUMMARY
    # ========================================================

    print(
        f"\nEpoch {epoch + 1:02d}/{EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Train Acc: {train_accuracy:.2f}%"
        f" | Test Loss: {test_loss:.4f}"
        f" | Test Acc: {test_accuracy:.2f}%"
        f" | Train: {train_images_per_sec:.1f} img/s"
        f" | Test: {test_images_per_sec:.1f} img/s"
        f" | Time: {epoch_elapsed:.2f}s"
        f"{best_marker}"
        f" | Nodes: {int(model.node_alive.sum().item())}"
        f" | Edges: {int(model.edge_alive.sum().item())}"
    )


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

print(f"Best Test Accuracy : {best_test_accuracy:.2f}%")
print(f"Best model saved   : best_model.pth")

print(
    f"Final Train Accuracy : "
    f"{history['train_accuracy'][-1]:.2f}%"
)

print(
    f"Final Test Accuracy  : "
    f"{history['test_accuracy'][-1]:.2f}%"
)

print(
    f"Final Train Speed    : "
    f"{history['train_images_per_sec'][-1]:.1f} images/sec"
)

print(
    f"Final Test Speed     : "
    f"{history['test_images_per_sec'][-1]:.1f} images/sec"
)

print("=" * 80)


Using device : cuda

[4096, 3000, 1500, 1000, 500, 300, 43]
[4096, 3000, 1500, 1000, 500, 300]
tensor([6836, 4720, 4819,  ..., 4581, 5620, 4938], device='cuda:0')
tensor([8498, 7576, 7206,  ..., 7654, 7269, 7467], device='cuda:0')
tensor([9554, 8787, 8851, 9519, 8790, 8614, 9140, 9249, 9560, 8633, 8806, 8709,
        9498, 9185, 8839, 8931, 8645, 9502, 8641, 8979, 9576, 9271, 9494, 8662,
        8747, 9423, 9237, 9291, 9425, 9097, 9163, 8904, 8727, 9103, 9446, 9322,
        8987, 9483, 8991, 8854, 9191, 9345, 9308, 8755, 8757, 9567, 9171, 8965,
        9343, 9334, 9262, 9476, 9215, 8639, 8860, 9090, 8713, 9426, 9323, 9036,
        9246, 9041, 9201, 9570, 9516, 8782, 9042, 9455, 8997, 9339, 9313, 9369,
        8973, 9592, 9264, 9211, 9154, 9051, 8957, 9189, 8986, 8770, 9052, 8836,
        9333, 9436, 9019, 9126, 9107, 9463, 8773, 8949, 9122, 8766, 9106, 9574,
        9165, 9433, 9564, 9101, 8967, 8640, 8666, 9316, 9020, 8956, 8616, 9261,
        8984, 9089, 8739, 8929, 9492, 8610, 8613

Epoch 01/2:   0%|          | 0/172 [00:00<?, ?batch/s]

Testing:   0%|          | 0/43 [00:00<?, ?batch/s]


Running adaptive topology optimization...
Accepted topology modification: {'type': 'shortcut', 'source': 10068, 'destination': 10177, 'weight': 0.1, 'utility': 0.9996966123580933}
Validation accuracy gain: 0.000%

Epoch 01/2 | Train Loss: 3.8468 | Train Acc: 2.33% | Test Loss: 3.8405 | Test Acc: 2.33% | Train: 31.1 img/s | Test: 84.7 img/s | Time: 6.04s ★ BEST | Nodes: 10439 | Edges: 1


Epoch 02/2:   0%|          | 0/172 [00:00<?, ?batch/s]

Testing:   0%|          | 0/43 [00:00<?, ?batch/s]


Running adaptive topology optimization...
Accepted topology modification: {'type': 'shortcut', 'source': 9260, 'destination': 9642, 'weight': 0.1, 'utility': 0.9992071390151978}
Validation accuracy gain: 0.000%

Epoch 02/2 | Train Loss: 3.8402 | Train Acc: 2.33% | Test Loss: 3.8372 | Test Acc: 2.33% | Train: 33.3 img/s | Test: 84.7 img/s | Time: 5.67s | Nodes: 10439 | Edges: 2

TRAINING COMPLETE
Best Test Accuracy : 2.33%
Best model saved   : best_model.pth
Final Train Accuracy : 2.33%
Final Test Accuracy  : 2.33%
Final Train Speed    : 33.3 images/sec
Final Test Speed     : 84.7 images/sec
